# Building a synthetic population by hand

This notebook manually creates a small population of agents, households, and zones.
It serves as a learning exercise to understand how the pieces fit together before
we automate population synthesis later.

## 1. Imports

We import the three core data classes from the `aibm` package:
- **Zone** — a spatial area with coordinates and land-use info
- **Household** — a group of agents sharing a home zone, vehicles, and income
- **Agent** — an individual person with demographic attributes

In [ ]:
from aibm import Agent, Household, Zone

## 2. Create zones

We define three zones that represent different parts of a small city.
Each zone has an id, a name, centroid coordinates (x, y), and a land-use dictionary
indicating what kinds of activity happen there.

In [ ]:
suburbs = Zone(
    id="Z1",
    name="Suburbs",
    x=4.85,
    y=52.35,
    land_use={"residential": True, "commercial": False, "education": False},
)

city_centre = Zone(
    id="Z2",
    name="City Centre",
    x=4.90,
    y=52.37,
    land_use={"residential": False, "commercial": True, "education": False},
)

uni_district = Zone(
    id="Z3",
    name="University District",
    x=4.88,
    y=52.36,
    land_use={"residential": True, "commercial": False, "education": True},
)

zones = [suburbs, city_centre, uni_district]
for z in zones:
    print(f"{z.name} ({z.id}): land_use={z.land_use}")

## 3. Create households and agents

Each household gets a `home_zone` id. When agents are added via `add_member()`,
they automatically inherit that zone — no need to set it on every agent manually.

### Household 1 — a working couple with a child (Suburbs)

In [ ]:
hh1 = Household(home_zone="Z1", num_vehicles=2, income_level="high")

alice = Agent(name="Alice", age=42, employment="employed", has_license=True)
bob = Agent(name="Bob", age=44, employment="employed", has_license=True)
charlie = Agent(name="Charlie", age=12, employment="unemployed", has_license=False)

for agent in [alice, bob, charlie]:
    hh1.add_member(agent)

print(f"{hh1} — home_zone={hh1.home_zone}, vehicles={hh1.num_vehicles}")

### Household 2 — a single young professional (City Centre)

In [ ]:
hh2 = Household(home_zone="Z2", num_vehicles=0, income_level="medium")

dana = Agent(name="Dana", age=28, employment="employed", has_license=True)
hh2.add_member(dana)

print(f"{hh2} — home_zone={hh2.home_zone}, vehicles={hh2.num_vehicles}")

### Household 3 — two students sharing a flat (University District)

In [ ]:
hh3 = Household(home_zone="Z3", num_vehicles=0, income_level="low")

eve = Agent(name="Eve", age=21, employment="student", has_license=False)
frank = Agent(name="Frank", age=23, employment="student", has_license=True)

for agent in [eve, frank]:
    hh3.add_member(agent)

print(f"{hh3} — home_zone={hh3.home_zone}, vehicles={hh3.num_vehicles}")

## 4. Inspect the population

Let's verify that everything is wired up correctly. The key thing to check is that
each agent's `home_zone` matches their household's zone — this was set automatically
by `add_member()`.

In [ ]:
households = [hh1, hh2, hh3]

for hh in households:
    print(f"\n{hh} | income={hh.income_level}, vehicles={hh.num_vehicles}")
    for agent in hh.members:
        print(
            f"  {agent.name:10s}  age={agent.age:3d}  "
            f"employment={agent.employment:12s}  "
            f"license={agent.has_license}  "
            f"home_zone={agent.home_zone}"
        )

## 5. Generate personas

Now we call `generate_persona()` on each agent. This sends the agent's demographics
and household context to the Gemini API, which returns a 1-2 sentence behavioural
profile describing travel habits and daily routine.

**Requires** the `GEMINI_API_KEY` environment variable to be set.

In [ ]:
from google import genai

# Create one shared client so we reuse the same connection for all agents,
# rather than letting generate_persona() create a new client each call.
client = genai.Client()

for hh in households:
    for agent in hh.members:
        agent.generate_persona(client=client, household=hh)
        print(f"{agent.name}: {agent.persona}\n")